# Data Wrangling for GenAI- the skills the rest of the course assumes

**Module 2 · Lesson 2.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Data formats for ML
- Polars + Pandas - when to use which
- Text wrangling - when the real world fights you
- PDF and document ingestion - the RAG step
- Schema validation + train/val/test splits

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
!pip install -q pandas scikit-learn langchain langchain-text-splitters polars tiktoken unstructured pydantic

# Reproducibility - the lesson uses seeded random values throughout

> 🍳 **Analogy**
>
> **Data wrangling is restocking a kitchen the night before service.** You do not cook during service; you cook DURING the wrangling. Get the ingredients (parse), prep them (clean), label them (validate), portion them (split), put them in the right containers (format). When service starts the chef just assembles. RAG ingestion, fine-tuning data prep, and eval-set curation are all variants of this kitchen restock.
> **Where this analogy breaks down:** A real kitchen has a finite menu - you know what dishes you are cooking. ML data prep often does not - the "menu" (your downstream task) keeps changing as the team learns. So good data wrangling produces re-usable building blocks (clean Parquet, schema-validated JSONL) NOT one-shot pipelines. The analogy also breaks down on waste: a chef tosses a bad onion, but in ML the "bad onion" (a malformed row) might be the most informative sample for your eval set. Don't filter aggressively; document and defer.

### Data formats for ML

CSV vs JSONL vs Parquet vs Arrow - the architecture choice

The format you store data in IS the architecture choice. A 4-million-row Parquet loads into memory in ~300ms; the same data as a 1.2GB CSV takes 12 seconds and uses 5x the RAM. A JSONL is what the OpenAI fine-tuning API wants; you cannot upload a CSV. Apache Arrow is the in-memory format that Polars and DuckDB share to skip serialization entirely. Picking the right format is the single highest-leverage decision you make as a GenAI data engineer.

| Format | Best for | Avoid when |
|---|---|---|
| CSV | Human-readable exports, downstream loaders that expect it | Nested data, embeddings, >1M rows, non-ASCII text |
| JSONL | Fine-tuning datasets, LLM-output logs, append-only streaming | Heavy querying (parse cost), data >1GB |
| Parquet | Production storage, columnar analytics, anything >100k rows | Tiny datasets (overhead), human inspection without tools |
| Apache Arrow | In-memory inter-process (Polars to DuckDB to PyTorch) | Long-term storage (use Parquet instead) |

**📝 `format-cost-demo.py`** - *DEMO*

*Illustrative snippet — reads external files/dependencies not created in this notebook; shown for reference (does not run on Restart & Run All).*

```python
import polars as pl

# Same 1M-row dataset, four formats, four cost profiles:
df = pl.read_csv('events.csv')         # 12.0s,  1.2 GB RAM, no schema
df = pl.read_ndjson('events.jsonl')    # 8.4s,   1.8 GB RAM, nested ok
df = pl.read_parquet('events.parquet') # 0.3s,   220 MB RAM, columnar
df = pl.read_ipc('events.arrow')       # 0.1s,   220 MB RAM, zero-copy

# Output:
# Parquet wins on every dimension except human-readability.
# CSV wins only when a downstream loader cannot read anything else.
```


> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> ⚠ The "just use CSV" trap (common misconception)
> "CSV is universal so I will just use it." **Actually**, CSV silently corrupts ML data in four ways: (1) UTF-8 BOM at start of file breaks loaders that do not handle it, (2) commas inside text fields require quoting which not all writers do correctly, (3) an integer column with one missing value becomes a float column (3.0 instead of 3), (4) nested structures get flattened into stringified Python. Every one of these has bitten a production fine-tuning run.

> 📦 **Info**
>
> ⚖ Format-by-stage cheat sheet (tradeoff)
> - **Raw ingestion** (APIs, scrapers, dumps): JSONL - append-only, schemaless
> - **Cleaned + canonical**: Parquet - columnar, compressed, fast
> - **Fine-tuning serving** (to OpenAI/Anthropic/HF): JSONL - the only format APIs accept
> - **Tabular ML training**: Parquet via Polars or pandas-via-pyarrow
> - **Production warehouse**: Parquet on S3 + DuckDB/Athena/BigQuery on top
> **Production note:** OpenAI fine-tuning API accepts JSONL only. HuggingFace Datasets uses Apache Arrow under the hood. Snowflake / BigQuery default to Parquet for external tables. ([OpenAI fine-tuning format](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset), [Apache Arrow](https://arrow.apache.org/))

> 📦 **Info**
>
> ❌ What NOT to do (anti-example)
> Store your fine-tuning training set as a CSV with the user prompt in column A and the response in column B. Newlines inside the response break the row. Commas in the prompt break the column. Long context overflows Excel's 32K-char cell limit if anyone opens it. The OpenAI API rejects it. **Right approach:** JSONL with one row per training example.

### Polars + Pandas - when to use which

The 2026 default is Polars; Pandas for ecosystem interop

> 📦 **Info**
>
> ⚠ The "Pandas is the default" trap (common misconception, opens the step)
> "Everyone knows Pandas, so it is the safe choice." **Actually**, Pandas was designed in a single-core single-threaded world. It loads everything eagerly. For >1M-row datasets - which is most production data - it is 5-~10x slower than Polars and uses 3-~5x more RAM. **The 2026 default for new code is Polars.** Pandas remains a compatibility layer for sklearn / many ML libs that expect a `pd.DataFrame`. Use it for last-mile interop, not for the heavy lifting.

Polars (Rust, 2020) is the modern Pandas replacement: lazy evaluation, columnar memory, multi-threaded by default. For GenAI data work in 2026, lead with Polars. Reach for Pandas only when an external library demands it.

**📝 `polars-vs-pandas.py`** - *DEMO*

*Illustrative snippet — reads external files/dependencies not created in this notebook; shown for reference (does not run on Restart & Run All).*

```python
# Same query, two libraries
import pandas as pd
df_pd = pd.read_parquet('logs.parquet')
slow = df_pd[df_pd['model'] == 'claude-sonnet-4-6'].groupby('user_id')['cost'].sum()
# Output: 8.2 seconds, 12GB RAM peak

import polars as pl
df_pl = pl.scan_parquet('logs.parquet')  # lazy
fast = (
    df_pl
    .filter(pl.col('model') == 'claude-sonnet-4-6')
    .group_by('user_id')
    .agg(pl.col('cost').sum())
    .collect()  # only NOW does it execute
)
# Output: 0.9 seconds, 2GB RAM peak
```


> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> ⚖ When to reach for which (tradeoff)
> **Pick Polars when:** data >100k rows, you will filter/group/join multiple times, you want lazy evaluation. **Pick Pandas when:** integrating with sklearn / a library with only Pandas accessors, single small file (<10k rows). **DuckDB SQL** is the third option (Lesson 2.2) that beats both on certain workloads.
> **Production note:** Polars is the default DataFrame at JPMorgan, Stripe, Booking.com, and DuckDB Labs. Anthropic Claude Memory uses Polars on Parquet. Apache Spark is for cluster-scale (>100GB single dataset); Polars covers up to single-machine ~100M rows. ([Polars docs](https://pola.rs/), [Polars on GitHub](https://github.com/pola-rs/polars))

> 📦 **Info**
>
> ❌ What NOT to do (anti-example)
> Load a 50M-row LLM log Parquet into Pandas with `pd.read_parquet`. Your laptop has 16GB RAM. Pandas tries to allocate ~50GB. Your machine starts swapping. **Right approach:** Polars `pl.scan_parquet()` + lazy filter + collect-only-at-end. Returns in <2s on the same data.

### Text wrangling - when the real world fights you

UTF-8 BOM, Unicode normalization, tokenizer-aware truncation

Real-world text data is hostile. UTF-8 BOM at the start of files. Mixed encodings within a corpus. Right-to-left text. Emoji that span multiple bytes. Zero-width spaces. The naive `str.strip()` will not save you. This step is the defensive-text-handling toolkit.

**📝 `text-defense.py`** - *DEMO*

In [ ]:
# Setup: write a small messy.txt so the text-defense demo below runs on Restart & Run All.
# It intentionally contains a UTF-8 BOM, zero-width chars, non-breaking spaces, and whitespace runs.
bom, zwsp, zwnj, nbsp, nl = chr(0xFEFF), chr(0x200B), chr(0x200C), chr(0xA0), chr(10)
messy = bom + '  Hello' + zwsp + '  world' + zwnj + nl + nl + '  cafe' + nbsp + nbsp + 'test    done  '
with open('messy.txt', 'w', encoding='utf-8') as _f:
    _f.write(messy)


In [ ]:
import unicodedata, re, tiktoken

# 1. Always read with explicit encoding + replace policy
with open('messy.txt', encoding='utf-8-sig', errors='replace') as f:
    text = f.read()
# utf-8-sig strips BOM; errors=replace substitutes bad bytes.

# 2. Normalize Unicode form (combining characters)
text = unicodedata.normalize('NFKC', text)

# 3. Strip zero-width characters (U+200B-200D, U+FEFF)
text = re.sub(r'[​-‏‪-‮⁠﻿]', '', text)

# 4. Collapse whitespace runs
text = re.sub(r'\s+', ' ', text).strip()

# 5. Tokenizer-aware truncation
enc = tiktoken.encoding_for_model('gpt-4o')
tokens = enc.encode(text)
text_8k = enc.decode(tokens[:8000])
# Output:
# 5 defensive transforms applied. Text is now safe for embedding / fine-tuning.

> 📦 **Info**
>
> ⚠ The "text is just strings" trap (common misconception)
> "Python handles Unicode, so I do not have to think about encoding." **Actually**, Python 3 handles Unicode WHEN you tell it the source encoding. Your CSV from Windows is Windows-1252. Your DB export is UTF-8 with BOM. Without explicit `encoding=` and `errors=`, you get `UnicodeDecodeError` at character 4827 of a 200MB file and your pipeline crashes. Always specify encoding; always normalize Unicode form; always cut at token boundaries.

> 📦 **Info**
>
> ⚖ Cleaning levels by use case (tradeoff)
> - **Embedding for retrieval:** light cleaning. Preserve case, punctuation, numbers - they encode meaning.
> - **Fine-tuning data:** heavier cleaning. Normalize Unicode, collapse whitespace, strip HTML.
> - **Tokenizer training:** minimal cleaning. Over-cleaning destroys signal.
> - **PII redaction:** use Presidio or scrubadub, not regex.
> **Production note:** [ftfy](https://github.com/rspeer/python-ftfy) fixes mangled unicode (used at LinkedIn, OpenAI). [tiktoken](https://github.com/openai/tiktoken) is OpenAI's tokenizer. [Presidio](https://github.com/microsoft/presidio) is Microsoft's PII redaction library used in HIPAA-compliant LLM pipelines.

> 📦 **Info**
>
> ❌ What NOT to do (anti-example)
> Truncate prompts to 4000 *characters* before sending to an LLM. Some will be truncated mid-emoji because the next char was the second byte of a 4-byte sequence - the API rejects the malformed UTF-8 with a 400 error you spend 2 hours debugging. **Right approach:** truncate by token count using `tiktoken` (GPT), `anthropic.tokenize()` (Claude), or HF `tokenizers` (open models).

### PDF and document ingestion - the RAG step

The 4-stage pipeline every RAG system needs

This is the step every RAG engineer fails on first. Naive intuition: "PyPDF2 can extract text, then I will chunk by length." Reality: PDFs are a visual layout format, not a text format. A 2-column research paper extracts as alternating columns of garbage. Headers and footers repeat on every page and pollute embeddings. This step builds the canonical 2026 RAG ingestion pipeline.

**📝 `rag-ingestion.py`** - *DEMO*

*Illustrative snippet — reads external files/dependencies not created in this notebook; shown for reference (does not run on Restart & Run All).*

```python
from unstructured.partition.auto import partition
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o')

# Stage 1: Parse - auto-route by file type
elements = partition(filename='paper.pdf', strategy='hi_res')
# hi_res uses layout-aware parsing for 2-column papers

# Stage 2: Filter - drop headers/footers, page numbers
content_elements = [
    el for el in elements
    if el.category not in ('Header', 'Footer', 'PageNumber')
]

# Stage 3: Chunk - token-based, with overlap
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    length_function=lambda t: len(enc.encode(t)),
    separators=['\n\n', '\n', '. ', ' ', '']
)
texts = splitter.split_text('\n\n'.join(el.text for el in content_elements))

# Stage 4: Attach metadata for vector DB insertion
chunks = [
    {'text': t, 'source': 'paper.pdf', 'chunk_id': f'chunk-{i:04d}',
     'token_count': len(enc.encode(t))}
    for i, t in enumerate(texts)
]
# Output:
# ~24 chunks per typical research paper, ready for embedding
```


> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> ⚠ The "just use PyPDF2" trap (common misconception)
> "PyPDF2 / pdfplumber is good enough for PDFs." **Actually**, both struggle on 2-column layouts (they read left-to-right across columns, producing garbage), cannot extract table structure, and do not preserve heading hierarchy. **2026 default is unstructured.io** which auto-routes by document type and uses layout-aware parsing. For born-digital PDFs only, PyMuPDF (fitz) is fastest. For tables, IBM Docling beats unstructured.io.

> 📦 **Info**
>
> ⚖ Document parser cheat-sheet (tradeoff)
> - **unstructured.io** (`strategy='hi_res'`): auto-routes, layout-aware. Default for production RAG.
> - **PyMuPDF (fitz)**: fastest, born-digital PDFs only.
> - **pdfplumber**: best for tables in PDFs.
> - **AWS Textract / Azure Form Recognizer**: OCR + structured extraction from scans.
> - **IBM Docling (2024)**: open-source layout + tables + formulas. Newer alternative.
> **Production note:** Pinecone Assistant uses unstructured.io. LlamaIndex defaults to it. Anthropic Claude's internal RAG uses Docling. AWS Bedrock Knowledge Base uses Textract. ([unstructured.io on GitHub](https://github.com/Unstructured-IO/unstructured), [Docling on GitHub](https://github.com/DS4SD/docling), [LangChain splitters](https://python.langchain.com/docs/concepts/text_splitters/))

> 📦 **Info**
>
> ❌ What NOT to do (anti-example)
> Parse a 2-column research paper with PyPDF2, get garbage interleaved text, chunk it at 1000 characters, embed into your vector DB. Retrieval returns "Background subject sec- tion 2.1 attention propose this hypothesis..." - fragmented sentences split across column boundaries. RAG quality drops 20-~40%. **The fix:** unstructured.io with `strategy='hi_res'`. One library swap; catastrophic consequences if missed. **This is the single biggest RAG failure mode in 2026.**

### Schema validation + train/val/test splits

The quality gate that prevents downstream disasters

The last ~20% of data-wrangling work catches ~80% of downstream bugs. Schema validation rejects malformed rows before they waste GPU time. Stratified splits prevent "my eval set has zero examples of the minority class." Time-aware splits prevent leakage when your task is non-iid (almost always true for production data).

**📝 `validate-and-split.py`** - *DEMO*

*Illustrative snippet — reads external files/dependencies not created in this notebook; shown for reference (does not run on Restart & Run All).*

```python
from pydantic import BaseModel, field_validator, ValidationError
import polars as pl
from sklearn.model_selection import train_test_split

# 1. Schema validation with pydantic v2
class FineTuningExample(BaseModel):
    messages: list[dict]
    system_prompt: str | None = None
    metadata: dict = {}

    @field_validator('messages')
    def validate_messages(cls, v):
        roles = [m.get('role') for m in v]
        if 'assistant' not in roles:
            raise ValueError('must contain assistant message')
        return v

# Validate
df = pl.read_ndjson('finetune-raw.jsonl')
valid, malformed = [], []
for row in df.iter_rows(named=True):
    try: valid.append(FineTuningExample(**row))
    except ValidationError as e: malformed.append({'row': row, 'errors': e.errors()})
# Output:
# valid: 9743, malformed: 257  (~2.~6% would have wasted ~$200-500)

# 2. Stratified split
train, temp = train_test_split(valid, test_size=0.2,
    stratify=[r.metadata.get('intent') for r in valid], random_state=42)
val, test = train_test_split(temp, test_size=0.5)

# 3. Time-aware split (for non-iid LLM logs)
sorted_df = pl.read_parquet('logs.parquet').sort('timestamp')
n = sorted_df.height
train_t = sorted_df.slice(0, int(n * 0.8))
val_t = sorted_df.slice(int(n * 0.8), int(n * 0.1))
test_t = sorted_df.slice(int(n * 0.9), int(n * 0.1))
```


> 📦 **Info**
>
> ⚠ The "random_state=42" trap (common misconception)
> "I'll do 80/10/10 with random_state=42 for reproducibility." **Actually**, random splits assume iid samples. Most production data is NOT iid: LLM logs are time-correlated, support tickets are user-correlated, prompt-templates evolve. A random split puts the same user's data in train AND test, which leaks information. **The fix:** stratified by class for classification; GroupKFold for user-correlated; time-aware for time-series. `random_state=42` is theater if your split strategy is wrong.

> 📦 **Info**
>
> ⚖ Split-by-task cheat sheet (tradeoff)
> - **Single-shot classification** (independent, balanced): random + stratify
> - **User-grouped data** (LLM logs, support tickets): GroupKFold - all of user X in ONE split
> - **Time-series / drift-prone**: time-aware cutoff; val EARLIER than test
> - **Imbalanced classes**: stratified k-fold, report mean + std across folds
> - **Pretraining corpus**: deduplicate (NEAR-dups not just exact) first
> **Production note:** [pydantic v2](https://docs.pydantic.dev/latest/) is used by FastAPI, OpenAI Python SDK, LangChain. HuggingFace's `datasets.Dataset.train_test_split()` does both random and stratified. ([sklearn split strategies](https://scikit-learn.org/stable/modules/cross_validation.html))

> 📦 **Info**
>
> ❌ What NOT to do (anti-example)
> Skip pydantic, upload 10K JSONL to OpenAI fine-tuning. Roughly 2-~5% are malformed (missing content, wrong role, oversize messages). Costs $200-500 + 4 hours of wall time. Either the job rejects everything OR completes silently on garbage and quality drops 5 points. Debug time: 2 days. **Right approach:** pydantic schema as a pre-upload CI check. Catches ~100% of structural bugs in <1 second.

> ✅ **Info**
>
> 🎯 What you can now do
> - Pick the right format for any GenAI data task (CSV / JSONL / Parquet / Arrow) and explain why
> - Process a 50M-row dataset on a laptop using Polars + lazy evaluation
> - Defend production text against UTF-8 / BOM / encoding nightmares with five named defenses
> - Build a 4-stage RAG ingestion pipeline (parse, filter, chunk, embed-ready) for a folder of PDFs
> - Catch malformed fine-tuning data with pydantic before it wastes GPU time
> - Do a stratified or time-aware split that doesn't leak data
> **Where this shows up next:**
> - **Lesson 2.2 (SQL for GenAI):** consumes the Parquet / JSONL produced here
> - **Module 8.2 (Document Processing):** the 4-stage ingestion pipeline + chunking strategies expand directly from this lesson
> - **Module 9.2 (Data Preparation):** the JSONL + pydantic schema becomes the canonical fine-tuning format
> - **Module 14.3 (Drift Detection):** time-aware split pattern + deduplication of training corpora reappears in eval-set rotation

## Quick Check

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Data Wrangling for GenAI- the skills the rest of the course assumes**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `Lesson_2.1_Practice.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-2.1.html` - regenerate this notebook whenever the source HTML is updated.*
